In [1]:
import matplotlib.gridspec as gridspec

def plot(header):
    colors = ['blue', 'green', 'red', 'cyan', 'magenta', 'yellow', 'orange', 'purple', 
              'brown', 'pink', 'gray', 'olive', 'teal', 'navy', 'lime', 'gold', 'violet', 'blue']
   
    all_dates = pd.date_range(start=df1['days'].min(), end=df1['days'].max())
    
    if header != 'time' and header != 'days':
        fig = plt.figure(figsize=(8, 6))
        gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1])  # 2 Zeilen, Plot 3/4 und Tabelle 1/4

        ax = plt.subplot(gs[0])  # Plot in der oberen Zeile
        
        # Plot der Daten
        x_values_per_day = df1.groupby('days')[header].sum()
        x_values_per_day = x_values_per_day.reindex(all_dates, fill_value=None)
        ax.plot(x_values_per_day, color=colors[df1.columns.get_loc(header)], label='Anzahl')

        # Zellen für die Tabelle
        x_values_per_hour = df1[header]
        cells_per_hour = [x_values_per_hour.mean(), x_values_per_hour.median(),
                          x_values_per_hour.std(), x_values_per_hour.min(), x_values_per_hour.max()]

        cells_per_day = [x_values_per_day.mean(), x_values_per_day.median(), 
                         x_values_per_day.std(), x_values_per_day.min(), x_values_per_day.max()]

        # Runde die Werte für die Tabelle
        cells_per_hour = [round(value, 2) for value in cells_per_hour]
        cells_per_day = [round(value, 2) for value in cells_per_day]

        # Tabellenüberschriften
        titles = ['Mittelwert', 'Median', 'Standardabw.', 'Min', 'Max']

        # Tabelle in der unteren Zeile
        ax_table = plt.subplot(gs[1])  # Leere Achse, um die Tabelle zu setzen
        ax_table.axis('off')  # Achsen ausschalten, da nur die Tabelle angezeigt wird

        # Tabelle erstellen
        table = ax_table.table(cellText=[cells_per_hour, cells_per_day], rowLabels=['pro Stunde', 'pro Tag'],
                               colLabels=titles, loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.5)

        # Achsentitel und Legende für den Plot
        ax.set_title(header)
        ax.set_xlabel('Tag')
        ax.set_ylabel('Überquerungen mit dem Fahrrad')
        ax.legend()

        # Plot anzeigen
        plt.tight_layout()
        plt.show()

In [2]:


import pandas as pd
import matplotlib.pyplot as plt
import numpy as npy

df1: pd.DataFrame()

def iterate(filename):
    for name in filename.columns:
        plot(name)

In [ ]:
import os

for filename in os.listdir('../Projektdatensaetze/2018'):
    if filename.endswith('.csv') and filename != 'standortinformationen_dauerzählsstellen.csv':
        print(filename)
        df: pd.DataFrame = pd.read_csv('../Projektdatensaetze/2018/' + filename, sep=';|,')
        df1 = df.copy()
        df.fillna(0, inplace = True)
        df1 = df1.rename(columns={df1.columns[0]: 'time'})
        df1 = df1.rename(columns={df1.columns[1]: 'sec'})
        df1['time'] += '/'+df1['sec']+':00'
        df1 = df1.drop(columns=['sec'])
        df1['time'] = pd.to_datetime(df1['time'], format='mixed')
        df1['days'] = df1['time'].dt.date
        iterate(df1)
    else:
        continue

In [4]:
from functions import *

In [ ]:
dataframes = []
for filename in os.listdir('../Projektdatensaetze/2018'):
    if filename.endswith('.csv') and filename != 'standortinformationen_dauerzählsstellen.csv':
        print(filename)
        df: pd.DataFrame = pd.read_csv('../Projektdatensaetze/2018/' + filename, sep=';|,')
        df1 = df.copy()
        df.fillna(0, inplace = True)
        df1 = df1.rename(columns={df1.columns[0]: 'time'})
        df1 = df1.rename(columns={df1.columns[1]: 'sec'})
        df1['time'] += '/'+df1['sec']+':00'
        df1 = df1.drop(columns=['sec'])
        df1['time'] = pd.to_datetime(df1['time'], format='mixed')
        df1['days'] = df1['time'].dt.date
        dataframes.append(df1)
        #print_maxima(df1)

In [13]:
for df in dataframes:
    all_dates = pd.date_range(start=df['days'].min(), end=df['days'].max())
    columns = df.columns[1: len(df.columns) - 1]
    x_values_per_day = df.groupby('days', dropna=False)[columns].agg(lambda x: x.sum(skipna=False))
    x_values_per_day = x_values_per_day.reindex(all_dates, fill_value=None)
    length=len(x_values_per_day.columns)
    #keys= x_values_per_day.columns[1:length-1]
    #cols = (x_values_per_day == 0).any()
    #zero_days = x_values_per_day.loc[:,cols]
    zeros = []
    groups = df.groupby('days')
    for column in x_values_per_day.columns:
        zero_crossings = []
        values = [name for name, gruppe in groups if (gruppe[column] == 0).all()]
        zero_crossings.append(column)
        zero_crossings.append(values)
        zeros.append(zero_crossings)
    print(zeros)

[['5.01 BN - Kennedybrücke (Nordseite) Radfahrer Ri. Innenstadt', []], ['5.01 BN - Kennedybrücke (Nordseite) Radfahrer Ri. Beuel', []]]
[['5.02 BN - Kennedybrücke (Südseite) Barometer', []], ['5.02 BN - Kennedybrücke (Südseite) IN', []], ['5.02 BN - Kennedybrücke (Südseite) Barometer Radfahrer Ri. Beuel', []]]
[['5.03 BN - Nordbrücke (Südseite)', []], ['5.03 BN - Nordbrücke (Südseite) Radfahrer Ri. Beuel', []], ['5.03 BN - Nordbrücke (Südseite) Radfahrer Ri. Auerberg', []]]
[['5.04 BN - Nordbrücke (Nordseite)', []], ['5.04 BN - Nordbrücke (Nordseite) Radfahrer Ri. Beuel', []], ['5.04 BN - Nordbrücke (Nordseite) Radfahrer Ri. Auerberg', []]]
[['5.05 BN - Südbrücke (Südseite)', []], ['5.05 BN - Südbrücke (Südseite) Radfahrer Ri. Gronau', []], ['5.05 BN - Südbrücke (Südseite) Radfahrer Ri. Ramersdorf', []]]
[['5.06 BN - Südbrücke (Nordseite)', []], ['5.06 BN - Südbrücke (Nordseite) Radfahrer Ri. Gronau', []], ['5.06 BN - Südbrücke (Nordseite) Radfahrer Ri. Ri. Ramersdorf', []]]
[['5.07 BN

In [15]:
for df in dataframes:
    all_dates = pd.date_range(start=df['days'].min(), end=df['days'].max())
    columns = df.columns[1: len(df.columns) - 1]
    x_values_per_day = df.groupby('days', dropna=False)[columns].agg(lambda x: x.sum(skipna=False))
    x_values_per_day = x_values_per_day.reindex(all_dates, fill_value=None)
    pd.set_option('display.max_colwidth', None)
    null_mask = x_values_per_day.isnull().any(axis=1)
    null_rows = x_values_per_day[null_mask]
    length = len(x_values_per_day.columns)
    keys = x_values_per_day.columns[1:length - 1]
    nulls = []
    groups = df.groupby('days')
    for column in x_values_per_day.columns:
        null_crossings = []
        values = [name for name, gruppe in groups if gruppe[column].isnull().all()]
        null_crossings.append(column)
        null_crossings.append(values)
        nulls.append(null_crossings)
    display(nulls)

[['5.01 BN - Kennedybrücke (Nordseite) Radfahrer Ri. Innenstadt', []],
 ['5.01 BN - Kennedybrücke (Nordseite) Radfahrer Ri. Beuel', []]]

[['5.02 BN - Kennedybrücke (Südseite) Barometer', []],
 ['5.02 BN - Kennedybrücke (Südseite) IN', []],
 ['5.02 BN - Kennedybrücke (Südseite) Barometer Radfahrer Ri. Beuel', []]]

[['5.03 BN - Nordbrücke (Südseite)', []],
 ['5.03 BN - Nordbrücke (Südseite) Radfahrer Ri. Beuel', []],
 ['5.03 BN - Nordbrücke (Südseite) Radfahrer Ri. Auerberg', []]]

[['5.04 BN - Nordbrücke (Nordseite)', []],
 ['5.04 BN - Nordbrücke (Nordseite) Radfahrer Ri. Beuel', []],
 ['5.04 BN - Nordbrücke (Nordseite) Radfahrer Ri. Auerberg', []]]

[['5.05 BN - Südbrücke (Südseite)', []],
 ['5.05 BN - Südbrücke (Südseite) Radfahrer Ri. Gronau', []],
 ['5.05 BN - Südbrücke (Südseite) Radfahrer Ri. Ramersdorf', []]]

...

[['5.11 BN - Brühler Straße', []],
 ['5.11 BN - Brühler Straße Radfahrer Ri. Süden', []],
 ['5.11 BN - Brühler Straße Radfahrer Ri. Norden', []]]

[['5.12 BN - Straßburger Weg', []],
 ['5.12 BN - Straßenburger Weg Radfahrer Ri. Süden', []],
 ['5.12 BN - Straßenburger Weg Radfahrer Ri. Norden', []]]

[['5.13 BN - Wilhelm-Spiritus-Ufer', []],
 ['5.13 BN - Wilhelm-Spiritus-Ufer Ri. Süden', []],
 ['5.13 BN - Wilhelm-Spiritus-Ufer Ri. Norden', []]]

[['5.14 BN - Mc Cloy Weg', []],
 ['5.14 BN - Mc Cloy Weg Ri. Remagen', []],
 ['5.14 BN - Mc Cloy Weg Ri. Bonn', []]]

[['5.15 BN - Weg auf Damm Neil', []],
 ['5.15 BN - Weg auf Damm Neil Ri. Norden', []],
 ['5.15 BN - Weg auf Damm Neil Ri. Süden', []]]